# Real Estate Price Predictor
## Build a Regression Model to Predict Home Prices

This notebook demonstrates how to build a machine learning model to predict real estate prices based on location, amenities, and market trends.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Load and Explore Real Estate Dataset

In [ ]:
# Generate sample real estate data
np.random.seed(42)
n_samples = 500

data = {
    'square_feet': np.random.randint(1000, 5000, n_samples),
    'bedrooms': np.random.randint(1, 6, n_samples),
    'bathrooms': np.random.uniform(1, 4, n_samples).round(1),
    'age': np.random.randint(0, 50, n_samples),
    'garage_spaces': np.random.randint(0, 4, n_samples),
    'lot_size': np.random.randint(2000, 20000, n_samples),
    'location_score': np.random.uniform(1, 10, n_samples).round(1),
    'neighborhood': np.random.choice(['Downtown', 'Suburbs', 'Rural'], n_samples),
    'amenities_count': np.random.randint(0, 10, n_samples),
    'market_trend': np.random.choice(['Rising', 'Stable', 'Declining'], n_samples),
}

df = pd.DataFrame(data)

# Generate realistic prices
base_price = 100000
price = (
    base_price +
    (df['square_feet'] * 100) +
    (df['bedrooms'] * 50000) +
    (df['bathrooms'] * 30000) +
    (df['location_score'] * 20000) +
    (df['amenities_count'] * 5000) +
    ((100 - df['age']) * 1000) +
    (df['garage_spaces'] * 15000) +
    (df['lot_size'] * 2) +
    np.random.normal(0, 50000, n_samples)
)

# Apply trend and neighborhood factors
trend_factors = {'Rising': 1.1, 'Stable': 1.0, 'Declining': 0.9}
neighborhood_factors = {'Downtown': 1.2, 'Suburbs': 1.0, 'Rural': 0.8}

price = price * df['market_trend'].map(trend_factors)
price = price * df['neighborhood'].map(neighborhood_factors)

df['price'] = np.maximum(price, 50000).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
print(df.head())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Display basic statistics
print('Data Statistics:')
print(df.describe())
print(f'\nData Info:')
print(df.info())
print(f'\nMissing Values:')
print(df.isnull().sum())

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Price distribution
axes[0, 0].hist(df['price'], bins=30, edgecolor='black')
axes[0, 0].set_title('Price Distribution')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

# Square feet distribution
axes[0, 1].hist(df['square_feet'], bins=30, edgecolor='black')
axes[0, 1].set_title('Square Feet Distribution')
axes[0, 1].set_xlabel('Square Feet')
axes[0, 1].set_ylabel('Frequency')

# Age distribution
axes[0, 2].hist(df['age'], bins=30, edgecolor='black')
axes[0, 2].set_title('Age Distribution')
axes[0, 2].set_xlabel('Age (years)')
axes[0, 2].set_ylabel('Frequency')

# Neighborhood distribution
df['neighborhood'].value_counts().plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Neighborhood Distribution')
axes[1, 0].set_ylabel('Count')

# Market trend distribution
df['market_trend'].value_counts().plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Market Trend Distribution')
axes[1, 1].set_ylabel('Count')

# Bedrooms distribution
df['bedrooms'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 2])
axes[1, 2].set_title('Bedrooms Distribution')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing and Feature Engineering

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Create new features
df_processed['price_per_sqft'] = df_processed['price'] / df_processed['square_feet']
df_processed['bed_bath_ratio'] = df_processed['bedrooms'] / (df_processed['bathrooms'] + 1)
df_processed['age_squared'] = df_processed['age'] ** 2
df_processed['total_rooms'] = df_processed['bedrooms'] + df_processed['bathrooms']
df_processed['luxury_score'] = (df_processed['location_score'] * 10 + df_processed['amenities_count'] * 10) / 20

print('New features created:')
print(df_processed[['price_per_sqft', 'bed_bath_ratio', 'age_squared', 'total_rooms', 'luxury_score']].head())

In [ ]:
# Correlation analysis
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
correlation_matrix = df_processed[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

# Show top correlations with price
print('\nTop Features Correlated with Price:')
print(correlation_matrix['price'].sort_values(ascending=False))

## 5. Handle Missing Values and Outliers

In [ ]:
# Check for missing values
print('Missing values:')
print(df_processed.isnull().sum())

# Fill any missing values (if any)
df_processed = df_processed.fillna(df_processed.mean(numeric_only=True))

print('\nMissing values after handling:')
print(df_processed.isnull().sum())

In [ ]:
# Detect and handle outliers using IQR method
def remove_outliers_iqr(data, column, threshold=1.5):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    return data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]

initial_rows = len(df_processed)
df_processed = remove_outliers_iqr(df_processed, 'price')
df_processed = remove_outliers_iqr(df_processed, 'square_feet')

removed_rows = initial_rows - len(df_processed)
print(f'Removed {removed_rows} outlier rows')
print(f'Dataset shape after outlier removal: {df_processed.shape}')

## 6. Encode Categorical Variables

In [ ]:
# Label encode categorical variables
le_neighborhood = LabelEncoder()
le_market = LabelEncoder()

df_processed['neighborhood_encoded'] = le_neighborhood.fit_transform(df_processed['neighborhood'])
df_processed['market_trend_encoded'] = le_market.fit_transform(df_processed['market_trend'])

print('Neighborhood Encoding:')
print(dict(zip(le_neighborhood.classes_, le_neighborhood.transform(le_neighborhood.classes_))))

print('\nMarket Trend Encoding:')
print(dict(zip(le_market.classes_, le_market.transform(le_market.classes_))))

# Drop original categorical columns and keep encoded versions
df_processed = df_processed.drop(['neighborhood', 'market_trend'], axis=1)

print(f'\nProcessed DataFrame shape: {df_processed.shape}')
print(df_processed.head())

## 7. Split Data into Training and Testing Sets

In [ ]:
# Prepare features and target
X = df_processed.drop('price', axis=1)
y = df_processed['price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print(f'Training set size: {X_train_scaled.shape}')
print(f'Test set size: {X_test_scaled.shape}')
print(f'\nFeatures: {list(X.columns)}')

## 8. Build and Train Regression Models

In [ ]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train models
trained_models = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f'{name} trained successfully!\n')

## 9. Evaluate Model Performance

In [ ]:
# Evaluate all models
results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    
    results[name] = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'MAPE': mape,
        'Predictions': y_pred
    }

# Create results DataFrame
results_df = pd.DataFrame({
    name: results[name] 
    for name in results 
    if name != 'Predictions'
}).T

print('Model Performance Comparison:')
print(results_df.round(4))

# Find best model
best_model_name = results_df['R2'].idxmax()
print(f'\n\nBest Model: {best_model_name} (R² = {results_df.loc[best_model_name, "R2"]:.4f})')

## 10. Make Predictions on New Data

In [ ]:
# Example: Make predictions on new properties
new_properties = pd.DataFrame([
    {
        'square_feet': 2500,
        'bedrooms': 4,
        'bathrooms': 2.5,
        'age': 15,
        'garage_spaces': 2,
        'lot_size': 8000,
        'location_score': 8.5,
        'amenities_count': 6,
        'neighborhood_encoded': 0,  # Downtown
        'market_trend_encoded': 2,  # Rising
    },
    {
        'square_feet': 1800,
        'bedrooms': 3,
        'bathrooms': 2.0,
        'age': 5,
        'garage_spaces': 1,
        'lot_size': 6000,
        'location_score': 7.0,
        'amenities_count': 5,
        'neighborhood_encoded': 1,  # Suburbs
        'market_trend_encoded': 1,  # Stable
    }
])

# Add derived features
new_properties['price_per_sqft'] = 150  # Placeholder
new_properties['bed_bath_ratio'] = new_properties['bedrooms'] / (new_properties['bathrooms'] + 1)
new_properties['age_squared'] = new_properties['age'] ** 2
new_properties['total_rooms'] = new_properties['bedrooms'] + new_properties['bathrooms']
new_properties['luxury_score'] = (new_properties['location_score'] * 10 + new_properties['amenities_count'] * 10) / 20

# Scale features
new_properties_scaled = scaler.transform(new_properties)

# Get best model
best_model = trained_models[best_model_name]
predictions = best_model.predict(new_properties_scaled)

print(f'Predictions using {best_model_name}:')
print(f'\nProperty 1 (4 bed, 2.5 bath, Downtown): ${predictions[0]:,.0f}')
print(f'Property 2 (3 bed, 2 bath, Suburbs): ${predictions[1]:,.0f}')

## 11. Visualize Results and Feature Importance

In [ ]:
# Actual vs Predicted plots for best model
best_pred = results[best_model_name]['Predictions']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, best_pred, alpha=0.6)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Actual vs Predicted ({best_model_name})')
axes[0].grid(True, alpha=0.3)

# Residuals plot
residuals = y_test - best_pred
axes[1].scatter(best_pred, residuals, alpha=0.6)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].set_title('Residuals Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance for Random Forest or Gradient Boosting
if best_model_name in ['Random Forest', 'Gradient Boosting']:
    feature_importance = best_model.feature_importances_
    importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': feature_importance
    }).sort_values('Importance', ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x='Importance', y='Feature')
    plt.title(f'Top 10 Feature Importance ({best_model_name})')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_model_name} does not have feature importance scores')

In [ ]:
# Model Performance Comparison Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# R² Scores
r2_scores = results_df['R2'].sort_values(ascending=False)
axes[0, 0].bar(range(len(r2_scores)), r2_scores.values)
axes[0, 0].set_xticks(range(len(r2_scores)))
axes[0, 0].set_xticklabels(r2_scores.index, rotation=45)
axes[0, 0].set_ylabel('R² Score')
axes[0, 0].set_title('R² Score Comparison')
axes[0, 0].grid(True, alpha=0.3)

# RMSE
rmse_scores = results_df['RMSE'].sort_values()
axes[0, 1].bar(range(len(rmse_scores)), rmse_scores.values)
axes[0, 1].set_xticks(range(len(rmse_scores)))
axes[0, 1].set_xticklabels(rmse_scores.index, rotation=45)
axes[0, 1].set_ylabel('RMSE ($)')
axes[0, 1].set_title('RMSE Comparison')
axes[0, 1].grid(True, alpha=0.3)

# MAE
mae_scores = results_df['MAE'].sort_values()
axes[1, 0].bar(range(len(mae_scores)), mae_scores.values)
axes[1, 0].set_xticks(range(len(mae_scores)))
axes[1, 0].set_xticklabels(mae_scores.index, rotation=45)
axes[1, 0].set_ylabel('MAE ($)')
axes[1, 0].set_title('MAE Comparison')
axes[1, 0].grid(True, alpha=0.3)

# MAPE
mape_scores = results_df['MAPE'].sort_values()
axes[1, 1].bar(range(len(mape_scores)), mape_scores.values)
axes[1, 1].set_xticks(range(len(mape_scores)))
axes[1, 1].set_xticklabels(mape_scores.index, rotation=45)
axes[1, 1].set_ylabel('MAPE')
axes[1, 1].set_title('MAPE Comparison')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

**Project Completed Successfully!**

### Key Findings:
- **Best Model**: {}
- **R² Score**: {:.4f}
- **RMSE**: ${:.2f}
- **MAE**: ${:.2f}

### Next Steps:
1. Collect real estate data from your target market
2. Retrain the model with actual data
3. Deploy the model for production use
4. Monitor model performance over time
5. Update the model periodically with new data

### Model Uses:
- Property valuation for real estate agents
- Investment analysis for buyers
- Market trend analysis
- Automated property appraisals
""".format(best_model_name, results_df.loc[best_model_name, 'R2'], 
            results_df.loc[best_model_name, 'RMSE'], 
            results_df.loc[best_model_name, 'MAE'])